In [ ]:
!fusermount -u /content/drive  # unmount if mounted
!rm -rf /content/drive         # remove the directory completely
from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
from PIL import Image, UnidentifiedImageError
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# ✅ 1️⃣ Data Paths
data_dir = '/content/drive/MyDrive/cucumber'  # Updae this path

# ✅ 2️⃣ Check & Remove Corrupt Images
def remove_corrupt_images(data_dir):
    corrupt_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError):
                print(f'Corrupt image found: {file_path}')
                corrupt_files.append(file_path)

    for file_path in corrupt_files:
        os.remove(file_path)
        print(f'Removed {file_path}')

remove_corrupt_images(data_dir)

# ✅ 3️⃣ Data Transform & Loading
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# ✅ 4️⃣ Model Definition
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# ✅ 5️⃣ Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ✅ 6️⃣ Training Loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}')

# ✅ 7️⃣ Save Trained Model
torch.save(model.state_dict(), 'cotton_disease_classifier.pth')

# ✅ 8️⃣ Inference Function
def predict(image_path):
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/10], Loss: 1.8005, Val Acc: 0.5119
Epoch [2/10], Loss: 1.0308, Val Acc: 0.5435
Epoch [3/10], Loss: 0.7311, Val Acc: 0.5673
Epoch [4/10], Loss: 0.5534, Val Acc: 0.5435
Epoch [5/10], Loss: 0.4835, Val Acc: 0.5488
Epoch [6/10], Loss: 0.4020, Val Acc: 0.5541
Epoch [7/10], Loss: 0.3849, Val Acc: 0.5646
Epoch [8/10], Loss: 0.3378, Val Acc: 0.5515
Epoch [9/10], Loss: 0.3487, Val Acc: 0.5567
Epoch [10/10], Loss: 0.3166, Val Acc: 0.5541


In [ ]:
# ✅ 7️⃣ Save Trained Model
torch.save(model.state_dict(), 'cucumber_disease_classifier.pth')


In [ ]:
import os
from PIL import Image, UnidentifiedImageError
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1️⃣ Data Paths
# =========================
data_dir = '/content/drive/MyDrive/cucumber'  # Update this path

# =========================
# 2️⃣ Remove Corrupt Images
# =========================
def remove_corrupt_images(data_dir):
    corrupt_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError):
                print(f'Corrupt image found: {file_path}')
                corrupt_files.append(file_path)

    for file_path in corrupt_files:
        os.remove(file_path)
        print(f'Removed {file_path}')

remove_corrupt_images(data_dir)

# =========================
# 3️⃣ Data Transform & Loading (Augmentation + Normalization)
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =========================
# 4️⃣ Model Definition
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# =========================
# 5️⃣ Loss & Optimizer
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 6️⃣ Training Loop
# =========================
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}')

# =========================
# 7️⃣ Save Trained Model
# =========================
torch.save(model.state_dict(), 'cucumber_disease_classifier.pth')

# =========================
# 8️⃣ Inference Function
# =========================
def predict(image_path):
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/20], Loss: 1.8257, Val Acc: 0.3984
Epoch [2/20], Loss: 1.3208, Val Acc: 0.4749
Epoch [3/20], Loss: 1.1234, Val Acc: 0.4406
Epoch [4/20], Loss: 0.9546, Val Acc: 0.4644
Epoch [5/20], Loss: 0.8195, Val Acc: 0.4828
Epoch [6/20], Loss: 0.7101, Val Acc: 0.4908
Epoch [7/20], Loss: 0.6554, Val Acc: 0.4934
Epoch [8/20], Loss: 0.6117, Val Acc: 0.4908
Epoch [9/20], Loss: 0.5619, Val Acc: 0.4934
Epoch [10/20], Loss: 0.5107, Val Acc: 0.5040
Epoch [11/20], Loss: 0.4924, Val Acc: 0.4987
Epoch [12/20], Loss: 0.4750, Val Acc: 0.4697
Epoch [13/20], Loss: 0.4501, Val Acc: 0.4960
Epoch [14/20], Loss: 0.4433, Val Acc: 0.4908
Epoch [15/20], Loss: 0.4106, Val Acc: 0.5224
Epoch [16/20], Loss: 0.4014, Val Acc: 0.5066
Epoch [17/20], Loss: 0.4041, Val Acc: 0.4960
Epoch [18/20], Loss: 0.3942, Val Acc: 0.5040
Epoch [19/20], Loss: 0.3723, Val Acc: 0.5013
Epoch [20/20], Loss: 0.3730, Val Acc: 0.5092


In [ ]:
!pip install imagehash

import os
from PIL import Image, UnidentifiedImageError
import imagehash
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim

# =======================
# 1️⃣ Paths
# =======================
data_dir = '/content/drive/MyDrive/cucumber'  # original dataset
clean_dir = '/content/drive/MyDrive/cucumber_cleaned'  # cleaned images
os.makedirs(clean_dir, exist_ok=True)

# =======================
# 2️⃣ Remove duplicates & corrupt images & multi-leaf/text images
# =======================
hashes = {}

import os
from PIL import Image, UnidentifiedImageError
import imagehash

hashes = {}

for root, _, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            img = Image.open(file_path)
            img.verify()  # check corrupt
            img = Image.open(file_path)  # reopen for hashing

            # Convert to RGB to handle RGBA / grayscale issues
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Compute perceptual hash
            h = imagehash.phash(img)
            if h in hashes:
                print(f"Duplicate skipped: {file_path}")
                continue
            else:
                hashes[h] = file_path

        except (UnidentifiedImageError, OSError, SyntaxError):
            print(f"Corrupt skipped: {file_path}")
            continue

        # Skip mostly empty images or text (grayscale extrema check)
        img_gray = img.convert("L")
        extrema = img_gray.getextrema()
        if extrema[1] - extrema[0] < 10:
            print(f"Empty/text image skipped: {file_path}")
            continue

        # Copy cleaned image to new folder with .jpg extension
        rel_path = os.path.relpath(file_path, data_dir)
        base_name = os.path.splitext(rel_path)[0] + ".jpg"  # force .jpg
        new_path = os.path.join(clean_dir, base_name)
        os.makedirs(os.path.dirname(new_path), exist_ok=True)

        # Save as JPEG
        img.save(new_path, format='JPEG')

print("✅ Dataset cleaning completed!")

# =======================
# 3️⃣ Aggressive Data Transform
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =======================
# 4️⃣ Prepare Dataset
# =======================
full_dataset = datasets.ImageFolder(root=clean_dir, transform=transform_train)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Replace val transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 5️⃣ EfficientNet Transfer Learning
# =======================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# =======================
# 6️⃣ Training Loop
# =======================
num_epochs = 15
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0,0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_acc = correct/total
    avg_loss = running_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_cotton_model.pth")
        print(f"✅ Saved Best Model with Acc: {best_acc:.4f}")

# =======================
# 7️⃣ Inference Function
# =======================
def predict(image_path):
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image = transform_val(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs,1)
    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 6.6 MB/s eta 0:00:00
✅ Dataset cleaning completed!


FileNotFoundError: Couldn't find any class folder in /content/drive/MyDrive/cucumber_cleaned.

In [ ]:
import os
from PIL import Image
from torchvision import transforms
import random

clean_dir = '/content/drive/MyDrive/cucumber_cleaned'
balanced_dir = '/content/drive/MyDrive/cucumber_balanced'
os.makedirs(balanced_dir, exist_ok=True)

# Aggressive augmentation pipeline
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
])

target_count = 500  # desired number of images per class

for class_name in os.listdir(clean_dir):
    class_path = os.path.join(clean_dir, class_name)
    if not os.path.isdir(class_path):
        continue

    balanced_class_path = os.path.join(balanced_dir, class_name)
    os.makedirs(balanced_class_path, exist_ok=True)

    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    num_images = len(images)

    # Copy original images first
    for f in images:
        src = os.path.join(class_path, f)
        dst = os.path.join(balanced_class_path, f)
        Image.open(src).save(dst)

    # Generate augmented images if needed
    while num_images < target_count:
        img_name = random.choice(images)
        img_path = os.path.join(class_path, img_name)
        img = Image.open(img_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        aug_img = augment(img)
        new_name = f"{os.path.splitext(img_name)[0]}_aug_{num_images}.jpg"
        aug_img.save(os.path.join(balanced_class_path, new_name))
        num_images += 1

print("✅ Balanced dataset created!")


✅ Balanced dataset created!


In [ ]:
import os
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim

# =======================
# 1️⃣ Paths
# =======================
balanced_dir = '/content/drive/MyDrive/cucumber_balanced'  # balanced dataset

# =======================
# 2️⃣ Data Transforms
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =======================
# 3️⃣ Prepare Dataset
# =======================
full_dataset = datasets.ImageFolder(root=balanced_dir, transform=transform_train)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Replace val transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 4️⃣ EfficientNet-B0 Transfer Learning
# =======================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# =======================
# 5️⃣ Training Loop
# =======================
num_epochs = 15
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_efficientnet_cotton.pth")
        print(f"✅ Saved Best Model with Acc: {best_acc:.4f}")

# =======================
# 6️⃣ Inference Function
# =======================
def predict(image_path):
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    transform_val = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
    image = transform_val(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs,1)
    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Epoch [1/15] Loss: 1.8418 Val Acc: 0.4740
✅ Saved Best Model with Acc: 0.4740
Epoch [2/15] Loss: 1.3675 Val Acc: 0.4580
Epoch [3/15] Loss: 1.1523 Val Acc: 0.5480
✅ Saved Best Model with Acc: 0.5480
Epoch [4/15] Loss: 0.9765 Val Acc: 0.5320
Epoch [5/15] Loss: 0.8105 Val Acc: 0.5680
✅ Saved Best Model with Acc: 0.5680
Epoch [6/15] Loss: 0.7153 Val Acc: 0.5940
✅ Saved Best Model with Acc: 0.5940
Epoch [7/15] Loss: 0.6365 Val Acc: 0.5920
Epoch [8/15] Loss: 0.5602 Val Acc: 0.5880
Epoch [9/15] Loss: 0.5158 Val Acc: 0.5960
✅ Saved Best Model with Acc: 0.5960
Epoch [10/15] Loss: 0.5272 Val Acc: 0.5920
Epoch [11/15] Loss: 0.5232 Val Acc: 0.6000
✅ Saved Best Model with Acc: 0.6000
Epoch [12/15] Loss: 0.4976 Val Acc: 0.6080
✅ Saved Best Model with Acc: 0.6080
Epoch [13/15] Loss: 0.5173 Val Acc: 0.5980
Epoch [14/15] Loss: 0.5074 Val Acc: 0.6000
Epoch [15/15] Loss: 0.4872 Val Acc: 0.5900


In [ ]:
import os
from PIL import Image
from torchvision import transforms
import random

# Paths
clean_dir = '/content/drive/MyDrive/cucumber_cleaned'
balanced_dir = '/content/drive/MyDrive/cucumber_balanced'
os.makedirs(balanced_dir, exist_ok=True)

# Aggressive augmentation
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
])

target_count = 500  # desired images per class

for class_name in os.listdir(clean_dir):
    class_path = os.path.join(clean_dir, class_name)
    if not os.path.isdir(class_path):
        continue

    # Create folder for balanced dataset
    balanced_class_path = os.path.join(balanced_dir, class_name)
    os.makedirs(balanced_class_path, exist_ok=True)

    # List all images
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    num_images = len(images)

    # Copy original images
    for f in images:
        src = os.path.join(class_path, f)
        dst = os.path.join(balanced_class_path, f)
        with Image.open(src) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            img.save(dst)

    # Generate augmented images if needed
    while num_images < target_count:
        img_name = random.choice(images)
        img_path = os.path.join(class_path, img_name)
        with Image.open(img_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            aug_img = augment(img)
            new_name = f"{os.path.splitext(img_name)[0]}_aug_{num_images}.jpg"
            aug_img.save(os.path.join(balanced_class_path, new_name))
        num_images += 1

    print(f"✅ {class_name}: Balanced to {num_images} images")

print("🎯 All classes balanced to 500 images!")


✅ cucumber_frost_damage: Balanced to 500 images
✅ cucumber_leaf_miner_disease: Balanced to 500 images
✅ cucumber_beetle: Balanced to 500 images
✅ cucumber_foot_and_collar_rot: Balanced to 500 images
✅ cucumber_iron_deficiency_disease: Balanced to 500 images
✅ cucumber_fusarium_wilt_disease: Balanced to 500 images
✅ cucumber_flea_beetles_disease: Balanced to 500 images
✅ cucumber_bacterial_wilt_disease: Balanced to 500 images
✅ cucumber_fall_armyworm: Balanced to 500 images
✅ cucumber_calcium_deficiency: Balanced to 500 images
🎯 All classes balanced to 500 images!


In [ ]:
import os
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim

# =======================
# 1️⃣ Paths
# =======================
balanced_dir = '/content/drive/MyDrive/cucumber_balanced'  # balanced dataset

# =======================
# 2️⃣ Data Transforms
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =======================
# 3️⃣ Prepare Dataset
# =======================
full_dataset = datasets.ImageFolder(root=balanced_dir, transform=transform_train)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Replace val transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 4️⃣ EfficientNet-B0 Transfer Learning
# =======================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# =======================
# 5️⃣ Training Loop
# =======================
num_epochs = 10
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_efficientnet_cucumber.pth")
        print(f"✅ Saved Best Model with Acc: {best_acc:.4f}")

# =======================
# 6️⃣ Inference Function
# =======================
def predict(image_path):
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    transform_val = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
    image = transform_val(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs,1)
    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 151MB/s]


Epoch [1/10] Loss: 1.6843 Val Acc: 0.5130
✅ Saved Best Model with Acc: 0.5130
Epoch [2/10] Loss: 1.2284 Val Acc: 0.5810
✅ Saved Best Model with Acc: 0.5810
Epoch [3/10] Loss: 1.0122 Val Acc: 0.6490
✅ Saved Best Model with Acc: 0.6490
Epoch [4/10] Loss: 0.8351 Val Acc: 0.6730
✅ Saved Best Model with Acc: 0.6730
Epoch [5/10] Loss: 0.6902 Val Acc: 0.6840
✅ Saved Best Model with Acc: 0.6840
Epoch [6/10] Loss: 0.5932 Val Acc: 0.7150
✅ Saved Best Model with Acc: 0.7150
Epoch [7/10] Loss: 0.5176 Val Acc: 0.7400
✅ Saved Best Model with Acc: 0.7400
Epoch [8/10] Loss: 0.4580 Val Acc: 0.7340
Epoch [9/10] Loss: 0.4325 Val Acc: 0.7380
Epoch [10/10] Loss: 0.4137 Val Acc: 0.7380


In [ ]:
import os

# Create folder if it doesn't exist
save_dir = "/content/drive/MyDrive/models"
os.makedirs(save_dir, exist_ok=True)

# Now save the model
torch.save(model.state_dict(), os.path.join(save_dir, "best_efficientnet_cucumber.pth"))
print("✅ Model saved successfully!")


✅ Model saved successfully!


In [ ]:
import os

data_dir = '/content/drive/MyDrive/cucumber_balanced'  # your cleaned dataset

label_counts = {}

for class_name in os.listdir(data_dir):
    class_path = os.path.join(data_dir, class_name)
    if os.path.isdir(class_path):
        num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        label_counts[class_name] = num_images

# Print counts
for label, count in label_counts.items():
    print(f"{label}: {count} images")


cucumber_frost_damage: 971 images
cucumber_leaf_miner_disease: 873 images
cucumber_beetle: 935 images
cucumber_foot_and_collar_rot: 929 images
cucumber_iron_deficiency_disease: 1069 images
cucumber_fusarium_wilt_disease: 1020 images
cucumber_flea_beetles_disease: 981 images
cucumber_bacterial_wilt_disease: 1027 images
cucumber_fall_armyworm: 961 images
cucumber_calcium_deficiency: 1118 images


In [ ]:
!du -h /content/drive/MyDrive | sort -rh | head -20


5.2G	/content/drive/MyDrive
1014M	/content/drive/MyDrive/Projects_AI_DS/bearing_fault
1014M	/content/drive/MyDrive/Projects_AI_DS
901M	/content/drive/MyDrive/cucumber_balanced
777M	/content/drive/MyDrive/cucumber
719M	/content/drive/MyDrive/cotton
571M	/content/drive/MyDrive/Audio_Speech_Actors_01-24
371M	/content/drive/MyDrive/CropGen_dataset
248M	/content/drive/MyDrive/cucumber_cleaned
247M	/content/drive/MyDrive/Colab Notebooks
246M	/content/drive/MyDrive/emotion/data_files
246M	/content/drive/MyDrive/emotion
143M	/content/drive/MyDrive/cucumber_balanced/cucumber_iron_deficiency_disease
126M	/content/drive/MyDrive/cotton/cotton_frost_damage
120M	/content/drive/MyDrive/cucumber/cucumber_frost_damage
115M	/content/drive/MyDrive/cucumber/cucumber_fusarium_wilt_disease
111M	/content/drive/MyDrive/cucumber_balanced/cucumber_frost_damage
102M	/content/drive/MyDrive/cucumber_balanced/cucumber_fusarium_wilt_disease
100M	/content/drive/MyDrive/cotton/cotton_fusarium_wilt_disease
99M	/content